In [60]:
import orjson as json
from pathlib import Path
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.cli import tqdm
import difflib
from itertools import zip_longest
import random
from rapidfuzz import fuzz

In [49]:
from sys import path as sys_path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys_path:
    sys_path.append(str(project_root))
data_dir = project_root/"data/"
outputs_dir = project_root/"outputs/"

 ## Process JSONL files

In [ ]:
migration_data_directory = Path(outputs_dir / "json/migration_jsonls/1-with-duplicate-info")
files = sorted(migration_data_directory.glob("*_migration_events.jsonl"))

In [51]:
random.seed(42)

In [ ]:
def page_key(record):
    """page id from a dict with book, page_n and side."""
    return (record["book"], record["page_n"], record["side"])

pages = defaultdict(list)

random_files = random.sample(files, 200)
files_with_duplicates = []
for file in tqdm(random_files):
    with open(file, "rb") as f:
        records = [json.loads(line) for line in f]

    if not any(record["duplicate"] for record in records):
        continue

    files_with_duplicates.append(file)
    for record in records:
        pages[page_key(record)].append(record)

random_files = files_with_duplicates

100%|██████████| 200/200 [00:37<00:00,  5.31it/s]


In [61]:
originals = {}

for key, events in pages.items():
    original_keys = set()
    duplicate_of = events[0].get("duplicate_of") or []
    if len(duplicate_of) > 0:
        original_keys.add(page_key(duplicate_of[0]))
    if original_keys:
        originals[key] = original_keys

print(f"Pages loaded: {len(pages)}")
print(f"Pages with duplicates: {len(originals)}")

Pages loaded: 78561
Pages with duplicates: 11133


In [62]:
def page_lines(key):
    """One string per event on the page, with row_data joined by ", "."""
    return [", ".join(map(str, event["row_data"])) for event in pages[key]]

def side_by_side(left, right, width=60):
    """Render two lists of lines in two columns. Markers: | changed, > only left, < only right."""
    rows = []
    matcher = difflib.SequenceMatcher(None, left, right)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        for a, b in zip_longest(left[i1:i2], right[j1:j2], fillvalue=""):
            if tag == "equal":
                marker = "="
            elif a and b and fuzz.ratio(a, b) > 65:
                marker = "≈"
            elif a and b:
                marker = "|"
            elif a:
                marker = ">"
            else:
                marker = "<"
            rows.append(f"{a[:width]:<{width}} {marker} {b[:width]}")
    return "\n".join(rows)

def compare_random_pages(n, width=80):
    sampled = random.sample(list(originals.keys()), min(n, len(originals)))

    for dup_key in sampled:
        duplicate = page_lines(dup_key)
        for orig_key in originals[dup_key]:
            print(f"{"ORIGINAL " + str(orig_key):<{width}} | DUPLICATE {dup_key}")
            print("-" * (width * 2 + 3))
            if orig_key not in pages:
                print("page not found in loaded files\n")
                continue

            original = page_lines(orig_key)
            print(side_by_side(original, duplicate, width=width) or "identical", "\n")
            print("fuzz.ratio:", fuzz.ratio("|".join(original), "|".join(duplicate)))
            print("-" * (width * 2 + 3) + "\n")

compare_random_pages(n=100)

ORIGINAL ('tampereen_tuomiokirkkoseurakunta_muuttaneet_1898-1903_ap', 282, 'left') | DUPLICATE ('tampereen_tuomiokirkkoseurakunta_muuttaneet_1898-1903_mko40-50', 299, 'left')
-------------------------------------------------------------------------------------------------------------------------------------------------------------------
, , , , , , , , ,                                                                ≈ , , , , , , , 542, 71., 
751, 14/9, 1/9, Infanti Ida Juhontt Nummela, Ruovesi, III, 586, , 1,             ≈ 751, 14/9, 10/9, Torpennr Ida Juhonti Muumala, Ruovesi, II, 58l, , 1, 
75, 10/9, 1/9, Its. tt Ida Maria Iieni, Messukylä, III, 575, , 1, tän k. alaik.  ≈ Y57, 10/9, 1/9, Its.tt Ida Maria Iieni, Mekukylä, III, 75, , 1, tänkahik
ys3, 1/9, 1/9, Joh. Anna Emilia Pelkonen, ", III, 632, , /,                      ≈ 953, 19/9, 11/9, Talv. Anna Emilia Pelkonen, Lovion, III, 627, , 1, 
754, 2, 1/9, Taloll. pä Frans Henrik Keppilä, Kylmäkoski, II, 49, , ,            ≈ 154, 14,